<a href="https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

## Lane: Refresh / Content Opportunity Scoring

This notebook builds a transparent hand-written baseline for prioritizing content pages for human review.

The rule is based on the ML-06 signal audit:

- CTR relative to position was CONFIRMED and is the main weakness signal.
- Visibility was MIXED but remains useful as an impact signal.
- Position volatility was MIXED and is used only as supporting context.
- Average position is used for fair CTR comparison and eligibility filtering.

The baseline is a decision-support ranking. It does not automatically edit content or claim that a refresh will cause recovery.

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy

In [2]:
import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import whoami

In [3]:
import subprocess
import sys

REPO_URL = "https://github.com/Zafar488/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True,
    )

os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN missing. Add it in the Colab Secrets panel."
    )

user = whoami(token=HF_TOKEN)

print(
    "Hugging Face login successful:",
    user["name"],
)

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_FACT = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DuckDB connection ready.")
print("Development month: March 2026")
print("Feature window: March 1–15")
print("Outcome window: March 16–31 — audit only")

Working directory: /content/flyrank-ml-internship
Hugging Face login successful: zafar4050
DuckDB connection ready.
Development month: March 2026
Feature window: March 1–15
Outcome window: March 16–31 — audit only


## Baseline Data Frame

One row represents one anonymized content page for one client.

The feature window is March 1–15, 2026.

The outcome window is loaded only to repeat two signal checks. It is never used in the baseline score, reason code, action label, or ranked CSV.

The baseline score may use only:

- feature-window impressions;
- feature-window clicks and CTR;
- feature-window average position;
- feature-window position volatility;
- CTR relative to comparable pages in the same position band.

In [4]:
page_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS feature_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS feature_clicks,

        AVG(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS feature_avg_position,

        STDDEV_SAMP(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS feature_position_volatility,

        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS feature_available_days,

        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-16'
                     AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS outcome_impressions,

        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-16'
                     AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS outcome_available_days

    FROM {MARCH_FACT}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Raw page rows:", f"{len(page_frame):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw page rows: 331,437


In [5]:
page_frame = page_frame[
    (page_frame["feature_impressions"] >= 100)
    & (page_frame["feature_available_days"] >= 5)
    & (page_frame["outcome_available_days"] >= 5)
].copy()

page_frame["feature_ctr"] = np.where(
    page_frame["feature_impressions"] > 0,
    page_frame["feature_clicks"]
    / page_frame["feature_impressions"],
    np.nan,
)

page_frame["feature_position_volatility"] = (
    page_frame["feature_position_volatility"]
    .fillna(
        page_frame["feature_position_volatility"].median()
    )
)

page_frame["feature_daily_impressions"] = (
    page_frame["feature_impressions"]
    / page_frame["feature_available_days"]
)

page_frame["outcome_daily_impressions"] = (
    page_frame["outcome_impressions"]
    / page_frame["outcome_available_days"]
)

# Evaluation only — never a baseline input
page_frame["is_declining_proxy"] = (
    page_frame["outcome_daily_impressions"]
    < 0.80 * page_frame["feature_daily_impressions"]
).astype(int)

print("Eligible page rows:", f"{len(page_frame):,}")
print(
    "Audit-only declining proxy rate:",
    round(page_frame["is_declining_proxy"].mean(), 3),
)

Eligible page rows: 76,201
Audit-only declining proxy rate: 0.345


# 1. My Rule and Its Reason Code

Before encoding the baseline, I repeat two small signal checks.

## Signal Check 1 — Visibility

Visibility is checked because FlyRank's quick-win and review-priority logic depends on whether enough exposure is at stake.

ML-06 found a MIXED relationship with decline. Therefore, visibility will be used as an impact weight, not as proof that a page is weak.

## Signal Check 2 — CTR Relative to Position

CTR relative to position is checked because FlyRank's CTR-review logic compares pages with similar ranking opportunities.

ML-06 found this signal CONFIRMED. Therefore, below-expected CTR will be the main weakness component in the baseline.

## Plain-Words Rule

A page receives a high review score when:

1. it has enough impressions to matter;
2. its CTR is below the median CTR of comparable pages in the same position band;
3. its ranking position has been relatively unstable.

CTR weakness receives the largest weight because it was the clearest confirmed signal.

Visibility receives a smaller impact weight.

Position volatility receives the smallest supporting weight.

## Score

`baseline_action_score =`

- 60% position-adjusted CTR weakness;
- 30% visibility percentile;
- 10% position-volatility percentile.

## Single reason code

`visible_below_expected_ctr`

## Action label

`review_title_meta_and_intent`

The score only ranks candidates for human review.

In [6]:
page_frame["visibility_bucket"] = pd.cut(
    page_frame["feature_impressions"],
    bins=[
        99,
        499,
        1_999,
        9_999,
        np.inf,
    ],
    labels=[
        "100-499",
        "500-1,999",
        "2,000-9,999",
        "10,000+",
    ],
    include_lowest=True,
)

visibility_check = (
    page_frame
    .groupby(
        "visibility_bucket",
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        decline_rate=("is_declining_proxy", "mean"),
        median_impressions=("feature_impressions", "median"),
    )
    .reset_index()
)

visibility_check["decline_rate_pct"] = (
    100 * visibility_check["decline_rate"]
)

visibility_check

,visibility_bucket,n,decline_rate,median_impressions,decline_rate_pct
0,100-499,34590,0.348714,224.0,34.871350
1,"500-1,999",26261,0.334488,922.0,33.448840
2,"2,000-9,999",13474,0.346593,3443.5,34.659344
3,"10,000+",1876,0.398721,14786.5,39.872068


### Visibility Verdict

**MIXED**

The decline rate was not consistently ordered across all visibility buckets.

Visibility will therefore represent potential impact rather than directly representing weakness.

In [7]:
# Operational CTR-review population
ctr_population = page_frame[
    (page_frame["feature_impressions"] >= 500)
    & (page_frame["feature_avg_position"] > 0)
    & (page_frame["feature_avg_position"] <= 20)
].copy()

ctr_population["position_band"] = pd.cut(
    ctr_population["feature_avg_position"],
    bins=[0, 3, 10, 20],
    labels=[
        "Top 3",
        "Page 1",
        "Page 2",
    ],
    include_lowest=True,
)

ctr_reference = (
    ctr_population
    .groupby(
        "position_band",
        observed=True,
    )["feature_ctr"]
    .median()
    .rename("expected_ctr_for_band")
)

ctr_population = ctr_population.merge(
    ctr_reference,
    left_on="position_band",
    right_index=True,
    how="left",
)

ctr_population["ctr_gap"] = (
    ctr_population["feature_ctr"]
    - ctr_population["expected_ctr_for_band"]
)

ctr_population["ctr_status"] = np.where(
    ctr_population["ctr_gap"] < 0,
    "Below band median",
    "At or above band median",
)

ctr_signal_check = (
    ctr_population
    .groupby(
        [
            "position_band",
            "ctr_status",
        ],
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        decline_rate=("is_declining_proxy", "mean"),
        median_ctr=("feature_ctr", "median"),
        median_ctr_gap=("ctr_gap", "median"),
    )
    .reset_index()
)

ctr_signal_check["decline_rate_pct"] = (
    100 * ctr_signal_check["decline_rate"]
)

ctr_signal_check

,position_band,ctr_status,n,decline_rate,median_ctr,median_ctr_gap,decline_rate_pct
0,Top 3,At or above band median,3119,0.184995,0.005789,0.002778,18.499519
1,Top 3,Below band median,3118,0.371392,0.001378,-0.001633,37.139192
2,Page 1,At or above band median,10489,0.239489,0.004399,0.002284,23.948899
3,Page 1,Below band median,10488,0.418478,0.000940,-0.001175,41.847826
4,Page 2,At or above band median,3413,0.192499,0.004673,0.002162,19.249927
5,Page 2,Below band median,3411,0.358839,0.000830,-0.001680,35.883905


In [8]:
ctr_signal_summary = (
    ctr_population
    .groupby(
        "ctr_status",
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        decline_rate=("is_declining_proxy", "mean"),
        median_ctr_gap=("ctr_gap", "median"),
    )
    .reset_index()
)

ctr_signal_summary["decline_rate_pct"] = (
    100 * ctr_signal_summary["decline_rate"]
)

ctr_signal_summary

,ctr_status,n,decline_rate,median_ctr_gap,decline_rate_pct
0,At or above band median,17021,0.220081,0.002218,22.008108
1,Below band median,17017,0.397896,-0.001350,39.789622


# 2. Build the Ranked Queue

The ranked queue is restricted to pages with:

- at least 500 feature-window impressions;
- an average position between 1 and 20;
- valid feature-window CTR;
- valid position-band reference values.

The score contains no outcome-window or target-derived input.

## Score components

### CTR weakness score

Measures how far the page's CTR falls below the median CTR of its position band.

The score is capped between 0 and 1.

### Visibility score

Uses percentile rank rather than raw impressions because impressions are heavy-tailed.

### Volatility score

Uses percentile rank and receives only a small weight because its ML-06 verdict was MIXED.

In [9]:
queue = ctr_population.copy()

# Relative weakness:
# 0 means not below expected CTR
# 1 means very far below expected CTR
queue["ctr_weakness_score"] = np.where(
    queue["expected_ctr_for_band"] > 0,
    (
        -queue["ctr_gap"]
        / queue["expected_ctr_for_band"]
    ),
    0.0,
)

queue["ctr_weakness_score"] = (
    queue["ctr_weakness_score"]
    .clip(lower=0, upper=1)
)

# Heavy-tail-safe percentile impact score
queue["visibility_score"] = (
    queue["feature_impressions"]
    .rank(
        method="average",
        pct=True,
    )
)

# Supporting instability percentile
queue["volatility_score"] = (
    queue["feature_position_volatility"]
    .rank(
        method="average",
        pct=True,
    )
)

queue["baseline_action_score"] = (
    100
    * (
        0.60 * queue["ctr_weakness_score"]
        + 0.30 * queue["visibility_score"]
        + 0.10 * queue["volatility_score"]
    )
)

queue["reason_code"] = "visible_below_expected_ctr"

queue["action_label"] = (
    "review_title_meta_and_intent"
)

queue["confidence_note"] = np.select(
    [
        (
            queue["ctr_weakness_score"] >= 0.75
        )
        & (
            queue["visibility_score"] >= 0.75
        ),
        queue["ctr_weakness_score"] >= 0.50,
    ],
    [
        "higher-confidence review candidate",
        "medium-confidence review candidate",
    ],
    default="lower-confidence review candidate",
)

queue = (
    queue
    .sort_values(
        [
            "baseline_action_score",
            "feature_impressions",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(
    1,
    len(queue) + 1,
)

print("Ranked queue rows:", f"{len(queue):,}")
queue.head()

Ranked queue rows: 34,038


,client_hash_id,content_hash_id,feature_impressions,feature_clicks,feature_avg_position,feature_position_volatility,feature_available_days,outcome_impressions,outcome_available_days,feature_ctr,...,ctr_gap,ctr_status,ctr_weakness_score,visibility_score,volatility_score,baseline_action_score,reason_code,action_label,confidence_note,rank
0,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83772.0,0.0,8.607910,12.407646,15,62.0,16,0.000000,...,-0.002115,Below band median,1.000000,0.999794,0.994154,99.935366,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,1
1,client_62f4a7e64f5e0096,content_1ff6231687184dec,12634.0,0.0,16.675512,12.485486,15,2246.0,16,0.000000,...,-0.002510,Below band median,1.000000,0.975674,0.994594,99.216170,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,2
2,client_62f4a7e64f5e0096,content_9e0a8a913953b8d3,12020.0,0.0,8.453676,9.402399,14,981.0,15,0.000000,...,-0.002115,Below band median,1.000000,0.973118,0.983372,99.027264,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,3
3,client_fef1a8f436438636,content_551a522809e0358c,7379.0,0.0,13.612519,8.968112,15,3976.0,16,0.000000,...,-0.002510,Below band median,1.000000,0.936189,0.980052,97.886186,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,4
4,client_20259bd6705d81d4,content_1c8b7d9f25c118ab,16368.0,1.0,13.709873,5.866464,15,7321.0,16,0.000061,...,-0.002449,Below band median,0.975664,0.985575,0.924937,97.356451,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,5


In [10]:
queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "confidence_note",
    "position_band",
    "feature_impressions",
    "feature_clicks",
    "feature_ctr",
    "expected_ctr_for_band",
    "ctr_gap",
    "ctr_weakness_score",
    "feature_avg_position",
    "feature_position_volatility",
    "visibility_score",
    "volatility_score",
]

ranked_queue = queue[queue_columns].copy()

ranked_queue.head(10)

,rank,client_hash_id,content_hash_id,baseline_action_score,reason_code,action_label,confidence_note,position_band,feature_impressions,feature_clicks,feature_ctr,expected_ctr_for_band,ctr_gap,ctr_weakness_score,feature_avg_position,feature_position_volatility,visibility_score,volatility_score
0,1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,99.935366,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 1,83772.0,0.0,0.000000,0.002115,-0.002115,1.000000,8.607910,12.407646,0.999794,0.994154
1,2,client_62f4a7e64f5e0096,content_1ff6231687184dec,99.216170,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 2,12634.0,0.0,0.000000,0.002510,-0.002510,1.000000,16.675512,12.485486,0.975674,0.994594
2,3,client_62f4a7e64f5e0096,content_9e0a8a913953b8d3,99.027264,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 1,12020.0,0.0,0.000000,0.002115,-0.002115,1.000000,8.453676,9.402399,0.973118,0.983372
3,4,client_fef1a8f436438636,content_551a522809e0358c,97.886186,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 2,7379.0,0.0,0.000000,0.002510,-0.002510,1.000000,13.612519,8.968112,0.936189,0.980052
4,5,client_20259bd6705d81d4,content_1c8b7d9f25c118ab,97.356451,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 2,16368.0,1.0,0.000061,0.002510,-0.002449,0.975664,13.709873,5.866464,0.985575,0.924937
5,6,client_20259bd6705d81d4,content_dd76ae989a61d4f6,97.327105,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 2,6054.0,0.0,0.000000,0.002510,-0.002510,1.000000,19.481478,12.387243,0.912862,0.994124
6,7,client_fef1a8f436438636,content_cc9732f0da1d8d2d,97.103972,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 2,5702.0,0.0,0.000000,0.002510,-0.002510,1.000000,17.375762,14.309322,0.904474,0.996974
7,8,client_73cda7b4e4f265ea,content_c9f840183215651b,97.037135,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 1,18421.0,0.0,0.000000,0.002115,-0.002115,1.000000,5.216858,3.244500,0.988865,0.737117
8,9,client_fef1a8f436438636,content_c9643f42e0fb5214,96.290522,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 1,11060.0,1.0,0.000090,0.002115,-0.002024,0.957247,8.800659,9.040149,0.968330,0.980581
9,10,client_20259bd6705d81d4,content_59a897a6ca5418df,95.691139,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 2,4504.0,0.0,0.000000,0.002510,-0.002510,1.000000,16.234793,7.569551,0.868191,0.964540


In [11]:
CSV_PATH = (
    OUTPUT_DIR
    / "baseline_action_score.csv"
)

ranked_queue.to_csv(
    CSV_PATH,
    index=False,
)

print(
    "Baseline queue written to:",
    CSV_PATH,
)

print(
    "CSV rows:",
    f"{len(ranked_queue):,}",
)

Baseline queue written to: work/outputs/baseline_action_score.csv
CSV rows: 34,038


In [12]:
top_20 = ranked_queue.head(20).copy()

metrics = {
    "assignment": "ML-07",
    "lane": "Refresh / Content Opportunity Scoring",
    "development_month": "2026-03",
    "feature_window": "2026-03-01 to 2026-03-15",
    "queue_rows": int(len(ranked_queue)),
    "top_20_mean_score": float(
        top_20["baseline_action_score"].mean()
    ),
    "top_20_median_impressions": float(
        top_20["feature_impressions"].median()
    ),
    "top_20_mean_ctr_weakness": float(
        top_20["ctr_weakness_score"].mean()
    ),
    "score_weights": {
        "ctr_weakness": 0.60,
        "visibility": 0.30,
        "position_volatility": 0.10,
    },
    "reason_code": "visible_below_expected_ctr",
    "action_label": "review_title_meta_and_intent",
}

METRICS_PATH = (
    OUTPUT_DIR
    / "baseline_metrics.json"
)

with open(
    METRICS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metrics,
        file,
        indent=2,
    )

print(
    "Metrics receipt written to:",
    METRICS_PATH,
)

Metrics receipt written to: work/outputs/baseline_metrics.json


# 3. Top-20 Review

I manually review the top 20 as recommendations rather than accepting the score automatically.

For each row, the review includes:

- action;
- reason code;
- confidence note;
- why the page appears in the queue;
- what could make the recommendation wrong.

The review does not expose client names, URLs, titles, or private queries.

In [13]:
top_20_review = ranked_queue.head(20).copy()

def build_why_note(row):
    return (
        f"{int(row['feature_impressions']):,} impressions, "
        f"position {row['feature_avg_position']:.1f}, "
        f"CTR {row['feature_ctr']:.4f} versus "
        f"band median {row['expected_ctr_for_band']:.4f}, "
        f"with CTR weakness score "
        f"{row['ctr_weakness_score']:.2f}."
    )


def build_wrong_note(row):
    possible_reasons = []

    if row["feature_position_volatility"] >= (
        ranked_queue[
            "feature_position_volatility"
        ].quantile(0.90)
    ):
        possible_reasons.append(
            "temporary ranking volatility"
        )

    if row["feature_ctr"] == 0:
        possible_reasons.append(
            "zero clicks may reflect query mix or SERP behaviour"
        )

    if row["feature_avg_position"] > 15:
        possible_reasons.append(
            "low CTR may be normal near the bottom of page two"
        )

    possible_reasons.extend(
        [
            "seasonality",
            "intent mismatch outside the title or metadata",
            "SERP features reducing clicks",
        ]
    )

    return "; ".join(possible_reasons[:3])


top_20_review["why_selected"] = (
    top_20_review.apply(
        build_why_note,
        axis=1,
    )
)

top_20_review["what_would_make_it_wrong"] = (
    top_20_review.apply(
        build_wrong_note,
        axis=1,
    )
)

top_20_review[
    [
        "rank",
        "action_label",
        "reason_code",
        "confidence_note",
        "why_selected",
        "what_would_make_it_wrong",
    ]
]

,rank,action_label,reason_code,confidence_note,why_selected,what_would_make_it_wrong
0,1,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"83,772 impressions, position 8.6, CTR 0.0000 v...",temporary ranking volatility; zero clicks may ...
1,2,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"12,634 impressions, position 16.7, CTR 0.0000 ...",temporary ranking volatility; zero clicks may ...
2,3,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"12,020 impressions, position 8.5, CTR 0.0000 v...",temporary ranking volatility; zero clicks may ...
3,4,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"7,379 impressions, position 13.6, CTR 0.0000 v...",temporary ranking volatility; zero clicks may ...
4,5,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"16,368 impressions, position 13.7, CTR 0.0001 ...",temporary ranking volatility; seasonality; int...
5,6,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"6,054 impressions, position 19.5, CTR 0.0000 v...",temporary ranking volatility; zero clicks may ...
6,7,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"5,702 impressions, position 17.4, CTR 0.0000 v...",temporary ranking volatility; zero clicks may ...
7,8,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"18,421 impressions, position 5.2, CTR 0.0000 v...",zero clicks may reflect query mix or SERP beha...
8,9,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"11,060 impressions, position 8.8, CTR 0.0001 v...",temporary ranking volatility; seasonality; int...
9,10,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"4,504 impressions, position 16.2, CTR 0.0000 v...",temporary ranking volatility; zero clicks may ...


In [14]:
for _, row in top_20_review.iterrows():
    print(
        f"Rank {int(row['rank'])}: "
        f"Action={row['action_label']} | "
        f"Reason={row['reason_code']} | "
        f"Confidence={row['confidence_note']} | "
        f"Why={row['why_selected']} | "
        f"Could be wrong if={row['what_would_make_it_wrong']}"
    )

Rank 1: Action=review_title_meta_and_intent | Reason=visible_below_expected_ctr | Confidence=higher-confidence review candidate | Why=83,772 impressions, position 8.6, CTR 0.0000 versus band median 0.0021, with CTR weakness score 1.00. | Could be wrong if=temporary ranking volatility; zero clicks may reflect query mix or SERP behaviour; seasonality
Rank 2: Action=review_title_meta_and_intent | Reason=visible_below_expected_ctr | Confidence=higher-confidence review candidate | Why=12,634 impressions, position 16.7, CTR 0.0000 versus band median 0.0025, with CTR weakness score 1.00. | Could be wrong if=temporary ranking volatility; zero clicks may reflect query mix or SERP behaviour; low CTR may be normal near the bottom of page two
Rank 3: Action=review_title_meta_and_intent | Reason=visible_below_expected_ctr | Confidence=higher-confidence review candidate | Why=12,020 impressions, position 8.5, CTR 0.0000 versus band median 0.0021, with CTR weakness score 1.00. | Could be wrong if=tem

## Top-20 Review Observation

The top-ranked pages combine substantial visibility with CTR below the median for comparable position bands.

These are reasonable review candidates because their search exposure is large enough to matter and their CTR weakness is measured relative to similar ranking positions.

However, every recommendation can still be wrong. Low CTR may reflect seasonality, search intent, query mix, SERP features, brand recognition, or measurement noise rather than a title, metadata, or content problem.

The action is therefore review, not automatic editing.

# 4. Weak Picks and Leakage Check

A weak pick is a page whose total score is driven more by visibility than by strong CTR weakness.

These pages may appear high because they have substantial impressions even though their CTR gap is relatively modest.

I inspect such cases separately so the queue does not hide its own weaknesses.

In [15]:
top_20_review["weak_pick"] = (
    (top_20_review["ctr_weakness_score"] < 0.50)
    | (
        top_20_review["visibility_score"]
        > top_20_review["ctr_weakness_score"]
    )
)

top_20_review["weak_pick_reason"] = np.select(
    [
        (
            top_20_review["ctr_weakness_score"]
            < 0.35
        ),
        (
            top_20_review["visibility_score"]
            > top_20_review["ctr_weakness_score"]
        ),
    ],
    [
        "CTR weakness is relatively small",
        "Score is influenced more by visibility than weakness",
    ],
    default="No obvious weak-pick condition",
)

weak_picks = top_20_review[
    top_20_review["weak_pick"]
][
    [
        "rank",
        "baseline_action_score",
        "feature_impressions",
        "ctr_weakness_score",
        "visibility_score",
        "volatility_score",
        "weak_pick_reason",
    ]
]

print(
    "Weak picks among top 20:",
    len(weak_picks),
)

weak_picks

Weak picks among top 20: 2


,rank,baseline_action_score,feature_impressions,ctr_weakness_score,visibility_score,volatility_score,weak_pick_reason
4,5,97.356451,16368.0,0.975664,0.985575,0.924937,Score is influenced more by visibility than we...
8,9,96.290522,11060.0,0.957247,0.968330,0.980581,Score is influenced more by visibility than we...


### Weak-Pick Interpretation

Weak picks are not necessarily incorrect. They are simply recommendations where the score may rely heavily on business impact rather than a strong weakness signal.

A reviewer should be especially cautious with these rows and check whether the CTR gap is operationally meaningful before recommending changes.

## Leakage Check

The score must contain no:

- outcome-window fields;
- declining labels;
- decline ratios;
- product scores;
- product flags;
- URLs, domains, titles, or raw private queries.

The outcome proxy was used only in the two signal checks. It was not included in any score component or exported queue column.

In [16]:
baseline_input_columns = {
    "feature_impressions",
    "feature_ctr",
    "expected_ctr_for_band",
    "ctr_gap",
    "feature_avg_position",
    "feature_position_volatility",
    "ctr_weakness_score",
    "visibility_score",
    "volatility_score",
}

forbidden_fields = {
    "outcome_impressions",
    "outcome_daily_impressions",
    "outcome_available_days",
    "is_declining_proxy",
    "decline_ratio",
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "refresh_flag",
}

leakage_overlap = sorted(
    baseline_input_columns.intersection(
        forbidden_fields
    )
)

exported_forbidden = sorted(
    set(ranked_queue.columns).intersection(
        forbidden_fields
    )
)

print(
    "Forbidden score inputs:",
    leakage_overlap,
)

print(
    "Forbidden exported columns:",
    exported_forbidden,
)

assert len(leakage_overlap) == 0
assert len(exported_forbidden) == 0

print("Leakage check passed.")

Forbidden score inputs: []
Forbidden exported columns: []
Leakage check passed.


In [17]:
print("ML-07 NOTEBOOK RECEIPT")
print("-" * 60)

print(
    "Queue rows:",
    f"{len(ranked_queue):,}",
)

print(
    "Top-20 mean score:",
    round(
        top_20_review[
            "baseline_action_score"
        ].mean(),
        3,
    ),
)

print(
    "Top-20 median impressions:",
    int(
        top_20_review[
            "feature_impressions"
        ].median()
    ),
)

print(
    "Top-20 mean CTR weakness:",
    round(
        top_20_review[
            "ctr_weakness_score"
        ].mean(),
        3,
    ),
)

print(
    "Weak picks in top 20:",
    int(
        top_20_review[
            "weak_pick"
        ].sum()
    ),
)

print(
    "Reason code:",
    ranked_queue["reason_code"].unique(),
)

print(
    "Action label:",
    ranked_queue["action_label"].unique(),
)

print(
    "CSV exists:",
    CSV_PATH.exists(),
)

print(
    "Metrics JSON exists:",
    METRICS_PATH.exists(),
)

print(
    "Leakage fields found:",
    leakage_overlap,
)

assert len(ranked_queue) > 0
assert len(top_20_review) == 20
assert ranked_queue["rank"].is_monotonic_increasing
assert ranked_queue["baseline_action_score"].is_monotonic_decreasing
assert ranked_queue["reason_code"].nunique() == 1
assert ranked_queue["action_label"].nunique() == 1
assert CSV_PATH.exists()
assert METRICS_PATH.exists()
assert len(leakage_overlap) == 0

print("\nML-07 checks passed.")

ML-07 NOTEBOOK RECEIPT
------------------------------------------------------------
Queue rows: 34,038
Top-20 mean score: 96.443
Top-20 median impressions: 9219
Top-20 mean CTR weakness: 0.997
Weak picks in top 20: 2
Reason code: ['visible_below_expected_ctr']
Action label: ['review_title_meta_and_intent']
CSV exists: True
Metrics JSON exists: True
Leakage fields found: []

ML-07 checks passed.


## Final Results Summary

This notebook creates a transparent baseline review queue using only feature-window measurements.

The main weakness signal is CTR below the median for comparable ranking positions. This received a CONFIRMED verdict in ML-06.

Visibility is converted into a percentile-based impact score because raw impressions are heavily skewed.

Position volatility is included with a small weight because its signal-audit result was MIXED.

The final score uses:

- 60% position-adjusted CTR weakness;
- 30% visibility percentile;
- 10% position-volatility percentile.

Every row receives one reason code:

`visible_below_expected_ctr`

Every row receives one action label:

`review_title_meta_and_intent`

The top 20 were reviewed with a skeptical note explaining what could make each recommendation wrong.

The baseline is intended for human decision support. It does not prove that a page requires editing or that an edit would cause recovery.

# Self-check

- [x] I checked two signals before building the rule.
- [x] Both signal tables show `n`.
- [x] I used one transparent baseline rule.
- [x] The score has one reason code.
- [x] The score has one action label.
- [x] The ranked queue was written to `work/outputs/baseline_action_score.csv`.
- [x] I wrote a metrics JSON receipt.
- [x] I reviewed every one of the top 20 rows.
- [x] Every top-20 row has an action, reason code, confidence note, and skeptical note.
- [x] I identified possible weak picks.
- [x] I used no future-window or label-derived score input.
- [x] I exported no product flags or private fields.
- [x] The notebook runs from top to bottom without errors.
- [x] My conclusions use observed, measured, directional, and decision-support language.